# Group names: Chi Kien, Kedar Hegde, Isaiah Austin, Alan Chavarin

In [1]:
%pip install -q pymupdf pillow easyocr
%pip install -q torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 0. Data Loading & Preprocessing
Loads NDA documents from multiple files, extracts text (with OCR fallback), and splits them into clause-level inputs.

In [2]:
import fitz  # PyMuPDF
import io
import numpy as np
import easyocr
from PIL import Image
import torch
import os

# Initialize OCR reader
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

def extract_text_from_pdf(file_path, zoom=2.0):
    """Extract embedded PDF text and fallback to OCR."""
    doc = fitz.open(file_path)
    text_parts = []

    for page in doc:
        embedded_text = page.get_text().strip()
        if embedded_text:
            text_parts.append(embedded_text)
            continue

        # OCR fallback
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_np = np.array(img)

        ocr_lines = ocr_reader.readtext(img_np, detail=0, paragraph=True)
        ocr_text = "\n".join(line.strip() for line in ocr_lines if line and line.strip())
        text_parts.append(ocr_text)

    return "\n".join(part for part in text_parts if part)

def load_all_pdfs(folder_path):
    texts = []
    filenames = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            path = os.path.join(folder_path, file)
            text = extract_text_from_pdf(path)
            texts.append(text)
            filenames.append(file)
    
    return texts, filenames


# Load datasets
original_texts, original_files = load_all_pdfs("./data-original")
altered_texts, altered_files = load_all_pdfs("./data-altered")

print(f"Loaded {len(original_texts)} original files")
print(f"Loaded {len(altered_texts)} altered files")

print("\nSample preview:\n", original_texts[0][:500])

Loaded 15 original files
Loaded 1 altered files

Sample preview:
 Copyright © 2020 NonDisclosureAgreement.com. All Rights Reserved. 
Page 1 of 2
NON-DISCLOSURE AGREEMENT (NDA) 
This Nondisclosure Agreement or ("Agreement") has been entered into on the date of 
______________________________ and is by and between: 
Party Disclosing Information: ______________________________ with a mailing address of 
____________________________________________________________ (“Disclosing Party”). 
Party Receiving Information: ______________________________ with a mailing add


In [3]:
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    return text.strip()

original_clean = [clean_text(t) for t in original_texts]
altered_clean = [clean_text(t) for t in altered_texts]

print(original_clean[0][:300])

Copyright  2020 NonDisclosureAgreement.com. All Rights Reserved. Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ______________________________ and is by and between: Party Disclosing Information: _________________________


In [4]:
def split_into_clauses(text, min_len=50):
    cleaned = text.strip()
    if not cleaned:
        return []

    normalized = re.sub(r"\s+(?=(?:\d+\.|[a-z]\)|Instructions:|APPENDIX\s+[A-Z]|Page\s+\d+\s+of\s+\d+))", "\n", cleaned)

    chunks = re.split(r"\n+", normalized)
    chunks = [c.strip() for c in chunks if len(c.strip()) > min_len]

    if len(chunks) <= 6:
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z\[])", cleaned)
        sentences = [s.strip() for s in sentences if s.strip()]

        grouped = []
        current = []
        current_len = 0
        for sentence in sentences:
            current.append(sentence)
            current_len += len(sentence)
            if current_len >= 170:
                grouped.append(" ".join(current))
                current = []
                current_len = 0
        if current:
            grouped.append(" ".join(current))

        fallback = [c.strip() for c in grouped if len(c.strip()) > min_len]
        if len(fallback) > len(chunks):
            return fallback

    return chunks


original_clauses = []
for text in original_clean:
    original_clauses.extend(split_into_clauses(text))

altered_clauses = []
for text in altered_clean:
    altered_clauses.extend(split_into_clauses(text))

print("Total original clauses:", len(original_clauses))
print("Sample clause:\n", original_clauses[0])

Total original clauses: 249
Sample clause:
 Copyright  2020 NonDisclosureAgreement.com. All Rights Reserved.


In [5]:
%pip uninstall -y -q transformers
%pip install -q "transformers[sentencepiece]" --upgrade --force-reinstall

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\alanc\\AppData\\Roaming\\Python\\Python313\\site-packages\\transformers\\models\\metaclip_2\\__init__.py'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.11.0+cu130
GPU available: True


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_text(text):
    input_text = "summarize: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=20
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 9347.47it/s]


In [8]:
for i in range(3):
    sample_clause = original_clauses[i][:500]
    print(f"\n--- CLAUSE {i+1} ---")
    print("ORIGINAL:\n", sample_clause)
    print("SUMMARY:\n", summarize_text(sample_clause))


--- CLAUSE 1 ---
ORIGINAL:
 Copyright  2020 NonDisclosureAgreement.com. All Rights Reserved.
SUMMARY:
 Copyright 2020 NonDisclosureAgreement.com. All rights reserved. Copyright 2020 NonDisclosureAgreement.com.

--- CLAUSE 2 ---
ORIGINAL:
 Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ______________________________ and is by and between: Party Disclosing Information: ______________________________ with a mailing address of ____________________________________________________________ (Disclosing Party). Party Receiving Information: ______________________________ with a mailing address of ____________________________________________________________ (Rece
SUMMARY:
 this nondisclosure agreement or ("Agreement") has been entered into on the date of ______________________________. party Disclosing Information: ______________________________ with a mailing address of _______________________________________________

# 1. Feature 1: Summarization

## Baseline
Uses T5-small for generic clause summarization.

## Improved
Uses prompt engineering and beam search for improved summaries.

## Evaluation
Compares outputs using ROUGE scores and qualitative examples.

In [9]:
def summarize_text_improved(text):
    input_text = "summarize key points: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=25,
        num_beams=4,           # better quality
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
print("=== FEATURE 1: SUMMARIZATION ===\n")

for i in range(5):  # test on multiple clauses
    clause = original_clauses[i][:500]
    
    print(f"\n--- CLAUSE {i+1} ---")
    print("ORIGINAL:\n", clause)
    print("\nBASELINE:\n", summarize_text(clause))
    print("\nIMPROVED:\n", summarize_text_improved(clause))

=== FEATURE 1: SUMMARIZATION ===


--- CLAUSE 1 ---
ORIGINAL:
 Copyright  2020 NonDisclosureAgreement.com. All Rights Reserved.

BASELINE:
 Copyright 2020 NonDisclosureAgreement.com. All rights reserved. Copyright 2020 NonDisclosureAgreement.com.

IMPROVED:
 Copyright 2020 NonDisclosureAgreement.com. All rights Reserved. Copyright 2020 NonDisclosureAgreement.com.

--- CLAUSE 2 ---
ORIGINAL:
 Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ______________________________ and is by and between: Party Disclosing Information: ______________________________ with a mailing address of ____________________________________________________________ (Disclosing Party). Party Receiving Information: ______________________________ with a mailing address of ____________________________________________________________ (Rece

BASELINE:
 this nondisclosure agreement or ("Agreement") has been entered into on the date of _________

In [11]:
%pip install -q rouge-score

Note: you may need to restart the kernel to use updated packages.


In [12]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

print("\n=== FEATURE 1: ROUGE SCORES ===\n")

baseline_scores = []
improved_scores = []

for i in range(5):
    clause = original_clauses[i][:500]
    
    baseline = summarize_text(clause)
    improved = summarize_text_improved(clause)
    
    # Using original clause as reference
    score_base = scorer.score(clause, baseline)
    score_improved = scorer.score(clause, improved)
    
    baseline_scores.append(score_base['rouge1'].fmeasure)
    improved_scores.append(score_improved['rouge1'].fmeasure)

print("Baseline ROUGE-1 Avg:", sum(baseline_scores)/len(baseline_scores))
print("Improved ROUGE-1 Avg:", sum(improved_scores)/len(improved_scores))


=== FEATURE 1: ROUGE SCORES ===

Baseline ROUGE-1 Avg: 0.5980821792586498
Improved ROUGE-1 Avg: 0.7438612044442341


# 2. Feature 2: Risk Detection

## Baseline (Zero-shot)
Uses a pre-trained model for risk classification.

## Improved (Logistic Regression)
Trains a classifier on clause data using a train/test split.

## Evaluation
Compares models using accuracy, classification report, and sample predictions.

In [13]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 8786.51it/s]


In [14]:
labels = ["high risk", "low risk"]

result = classifier(sample_clause, candidate_labels=labels)

print(result)

{'sequence': '1. Definition of Confidential Information. For purposes of this Agreement, "Confidential Information" shall include all information or material that has or could have commercial value or other utility in the business in which Disclosing Party is engaged. If Confidential Information is in written form, the Disclosing Party shall label or stamp the materials with the word "Confidential" or some similar warning. If Confidential Information is transmitted orally, the Disclosing Party shall promptly ', 'labels': ['high risk', 'low risk'], 'scores': [0.5077658295631409, 0.49223417043685913]}


In [15]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [16]:
def label_clause(text):
    text = text.lower()
    if any(word in text for word in ["liability", "penalty", "terminate", "breach", "confidential"]):
        return "high risk"
    return "low risk"

texts = original_clauses
labels = [label_clause(c) for c in texts]

print("Total samples:", len(texts))
print("Sample labels:", labels[:10])

Total samples: 249
Sample labels: ['low risk', 'high risk', 'high risk', 'high risk', 'high risk', 'high risk', 'low risk', 'low risk', 'low risk', 'low risk']


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.3,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 174
Test size: 75


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model_lr.predict(X_test_vec)

print("=== LOGISTIC REGRESSION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

=== LOGISTIC REGRESSION RESULTS ===
Accuracy: 0.8933333333333333

Classification Report:
               precision    recall  f1-score   support

   high risk       0.97      0.82      0.89        38
    low risk       0.84      0.97      0.90        37

    accuracy                           0.89        75
   macro avg       0.90      0.89      0.89        75
weighted avg       0.90      0.89      0.89        75



In [20]:
print("\n=== ZERO-SHOT MODEL RESULTS ===")

zero_preds = []

for clause in X_test:
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    zero_preds.append(result["labels"][0])

print("Zero-shot Accuracy:", accuracy_score(y_test, zero_preds))


=== ZERO-SHOT MODEL RESULTS ===


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot Accuracy: 0.5466666666666666


In [21]:
print("\n=== SAMPLE PREDICTION COMPARISON ===")

for i in range(3):  # just show a few examples
    clause = X_test[i]

    print(f"\n--- CLAUSE {i+1} ---")
    print("TEXT:", clause[:200])

    # Zero-shot
    zero_result = classifier(clause, candidate_labels=["high risk", "low risk"])
    zero_pred = zero_result["labels"][0]

    # Logistic Regression
    lr_pred = model_lr.predict(vectorizer.transform([clause]))[0]

    print("Zero-shot:", zero_pred)
    print("Logistic Regression:", lr_pred)


=== SAMPLE PREDICTION COMPARISON ===

--- CLAUSE 1 ---
TEXT: 1. Either Party may disclose Confidential Information to the other Party in confidence provided that the disclosing Party identifies such information as proprietary and confidential either by marking 
Zero-shot: low risk
Logistic Regression: high risk

--- CLAUSE 2 ---
TEXT: 5. Relationships. Nothing contained in this Agreement shall be deemed to constitute either party a partner, joint venture or employee of the other party for any purpose.
Zero-shot: low risk
Logistic Regression: low risk

--- CLAUSE 3 ---
TEXT: XIII. Binding Arrangement. This Agreement will be binding upon and inure to the benefit of the parties hereto and each Partys respective successors and assigns. XIV. Severability.
Zero-shot: high risk
Logistic Regression: low risk


# 3. Feature 3: Jargon Explanation

## Baseline
Uses a simple prompt for explanation.

## Improved
Uses a refined prompt and decoding for clearer outputs.

## Comparison
Outputs are compared qualitatively to assess improvement and limitations.

In [22]:
def explain_clause_v1(text):
    input_text = "explain in simple terms: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [23]:
print("=== FEATURE 3: JARGON EXPLANATION ===\n")

for i in range(5):
    clause = original_clauses[i][:500]
    
    print(f"\n--- CLAUSE {i+1} ---")
    print("ORIGINAL:\n", clause)
    print("\nBASELINE EXPLANATION:\n", explain_clause_v1(clause))

=== FEATURE 3: JARGON EXPLANATION ===


--- CLAUSE 1 ---
ORIGINAL:
 Copyright  2020 NonDisclosureAgreement.com. All Rights Reserved.

BASELINE EXPLANATION:
  Copyright 2020 NonDisclosureAgreement.com. All Rights Reserved. Copyright 2020 NonDisclosureAgreement.com. All Rights Reserved.

--- CLAUSE 2 ---
ORIGINAL:
 Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ______________________________ and is by and between: Party Disclosing Information: ______________________________ with a mailing address of ____________________________________________________________ (Disclosing Party). Party Receiving Information: ______________________________ with a mailing address of ____________________________________________________________ (Rece

BASELINE EXPLANATION:
 : Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ________________________________

In [24]:
def explain_clause_v2(text):
    input_text = "rewrite this legal sentence in very simple plain English for a non-lawyer: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40,
        num_beams=5,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [25]:
print("\n=== IMPROVED EXPLANATIONS ===\n")

for i in range(5):
    clause = original_clauses[i][:500]
    
    print(f"\n--- CLAUSE {i+1} ---")
    print("IMPROVED EXPLANATION:\n", explain_clause_v2(clause))


=== IMPROVED EXPLANATIONS ===


--- CLAUSE 1 ---
IMPROVED EXPLANATION:
 rewrite this legal sentence in very simple plain English for a non-lawyer: Copyright 2020 NonDisclosureAgreement.com. All Rights Reserved. Copyright 2020 NonDisclosureAgreement.com. All Rights Reserved.

--- CLAUSE 2 ---
IMPROVED EXPLANATION:
 Page 1 of 2 NON-DISCLOSURE AGREEMENT (NDA) This Nondisclosure Agreement or ("Agreement") has been entered into on the date of ____________________________________________ and is by and between: Party Disclosing Information: _____________________________________ with a mailing address of ____________________________________________________________ (Disclosing Party). Party Receiving Information: _____________________________________ with a mailing address of 

--- CLAUSE 3 ---
IMPROVED EXPLANATION:
 sentence in very simple plain English for a non-lawyer: 1. Definition of Confidential Information. For purposes of this Agreement, "Confidential Information" shall include all inf

# 4. Feature 4: Version Comparison and New Risk Detection

## Goal
Compare two versions of an NDA, highlight clause-level changes, and flag newly introduced risks.

## Baseline
Word n-gram TF-IDF cosine similarity with greedy one-to-one matching, plus structural linking for likely merge/split cases.

## Improved
Hybrid similarity: combine word TF-IDF with character WB n-grams (better for OCR noise and light paraphrase), then the same matching pipeline with tuned thresholds.

## Approach
- Match clauses across versions using similarity scores.
- Categorize changes as added, removed, modified, or structural merge/split.
- Run risk classification on changed clauses and identify risk increases.
- Print baseline vs improved summaries for comparison.


In [26]:
import numpy as np
from difflib import unified_diff
from sklearn.metrics.pairwise import cosine_similarity


def predict_risk_label(clause):
    """Use the existing zero-shot classifier for stable risk labels."""
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    return result["labels"][0], float(result["scores"][0])


def _compare_clause_sets_from_similarity(
    sim,
    old_clauses,
    new_clauses,
    modified_threshold,
    candidate_floor,
    structural_threshold,
    diagnostics_extra=None,
):
    """Shared greedy matching + structural linking from a precomputed similarity matrix."""
    if not old_clauses and not new_clauses:
        return [], [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not old_clauses:
        return new_clauses, [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not new_clauses:
        return [], old_clauses, [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}

    candidate_pairs = []
    for new_idx in range(sim.shape[0]):
        row = sim[new_idx]
        best_old_idx = int(row.argmax())
        best_score = float(row[best_old_idx])
        candidate_pairs.append((best_score, new_idx, best_old_idx))

        for old_idx, score in enumerate(row):
            score = float(score)
            if score >= candidate_floor and old_idx != best_old_idx:
                candidate_pairs.append((score, new_idx, old_idx))

    candidate_pairs.sort(reverse=True, key=lambda x: x[0])

    matched_new = set()
    matched_old = set()
    modified = []
    unchanged = []

    for score, new_idx, old_idx in candidate_pairs:
        if new_idx in matched_new or old_idx in matched_old:
            continue
        if score < modified_threshold:
            continue

        matched_new.add(new_idx)
        matched_old.add(old_idx)

        old_clause = old_clauses[old_idx]
        new_clause = new_clauses[new_idx]
        if old_clause.strip() == new_clause.strip():
            unchanged.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })
        else:
            modified.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })

    unmatched_new_idx = [i for i in range(len(new_clauses)) if i not in matched_new]
    unmatched_old_idx = [i for i in range(len(old_clauses)) if i not in matched_old]

    structural_changes = []
    structural_new = set()
    structural_old = set()

    for new_idx in unmatched_new_idx:
        links = []
        for old_idx in unmatched_old_idx:
            score = float(sim[new_idx, old_idx])
            if score >= structural_threshold:
                links.append({"old_idx": old_idx, "similarity": score})

        if links:
            links = sorted(links, key=lambda x: x["similarity"], reverse=True)[:3]
            structural_changes.append(
                {
                    "new_clause": new_clauses[new_idx],
                    "linked_old_clauses": [
                        {"old_clause": old_clauses[item["old_idx"]], "similarity": item["similarity"]}
                        for item in links
                    ],
                }
            )
            structural_new.add(new_idx)
            for item in links:
                structural_old.add(item["old_idx"])

    added = [new_clauses[i] for i in unmatched_new_idx if i not in structural_new]
    removed = [old_clauses[i] for i in unmatched_old_idx if i not in structural_old]

    diagnostics = {
        "matched_pairs": len(matched_new),
        "candidate_pairs": len(candidate_pairs),
        "threshold": modified_threshold,
        "candidate_floor": candidate_floor,
        "structural_threshold": structural_threshold,
    }
    if diagnostics_extra:
        diagnostics.update(diagnostics_extra)

    return added, removed, modified, unchanged, structural_changes, diagnostics


def compare_clause_sets(
    old_clauses,
    new_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
):
    """
    Baseline: word n-gram TF-IDF cosine similarity + greedy one-to-one matching.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(ngram_range=(1, 2))
    old_X = vec.fit_transform(old_clauses)
    new_X = vec.transform(new_clauses)
    sim = cosine_similarity(new_X, old_X)

    return _compare_clause_sets_from_similarity(
        sim,
        old_clauses,
        new_clauses,
        modified_threshold,
        candidate_floor,
        structural_threshold,
        diagnostics_extra={"matcher": "baseline_tfidf_word"},
    )


def compare_clause_sets_improved(
    old_clauses,
    new_clauses,
    modified_threshold=0.50,
    candidate_floor=0.28,
    structural_threshold=0.36,
    word_char_weight=0.58,
):
    """
    Improved: blend word TF-IDF with character WB n-grams (more robust to OCR noise
    and light paraphrase), then same greedy matching pipeline.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec_w = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=1,
        max_df=0.95,
    )
    vec_c = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        max_df=0.98,
    )

    old_w = vec_w.fit_transform(old_clauses)
    new_w = vec_w.transform(new_clauses)
    old_c = vec_c.fit_transform(old_clauses)
    new_c = vec_c.transform(new_clauses)

    sim_w = cosine_similarity(new_w, old_w)
    sim_c = cosine_similarity(new_c, old_c)
    sim = word_char_weight * sim_w + (1.0 - word_char_weight) * sim_c
    sim = np.clip(sim, 0.0, 1.0)

    return _compare_clause_sets_from_similarity(
        sim,
        old_clauses,
        new_clauses,
        modified_threshold,
        candidate_floor,
        structural_threshold,
        diagnostics_extra={
            "matcher": "improved_hybrid_word_char_tfidf",
            "word_char_weight": word_char_weight,
        },
    )


def clause_word_diff(old_clause, new_clause, context_lines=1):
    old_words = old_clause.split()
    new_words = new_clause.split()
    diff_lines = list(
        unified_diff(old_words, new_words, fromfile="old", tofile="new", lineterm="", n=context_lines)
    )
    return "\n".join(diff_lines)


In [27]:
def analyze_version_changes(
    old_clauses,
    new_clauses,
    hybrid=False,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
    word_char_weight=0.58,
):
    if hybrid:
        added, removed, modified, unchanged, structural_changes, diagnostics = compare_clause_sets_improved(
            old_clauses,
            new_clauses,
            modified_threshold=modified_threshold,
            candidate_floor=candidate_floor,
            structural_threshold=structural_threshold,
            word_char_weight=word_char_weight,
        )
    else:
        added, removed, modified, unchanged, structural_changes, diagnostics = compare_clause_sets(
            old_clauses,
            new_clauses,
            modified_threshold=modified_threshold,
            candidate_floor=candidate_floor,
            structural_threshold=structural_threshold,
        )

    added_risks = []
    for clause in added:
        label, score = predict_risk_label(clause)
        if label == "high risk":
            added_risks.append({
                "clause": clause,
                "risk_label": label,
                "risk_score": score,
            })

    modified_risk_deltas = []
    for item in modified:
        old_label, old_score = predict_risk_label(item["old_clause"])
        new_label, new_score = predict_risk_label(item["new_clause"])

        is_increase = (old_label == "low risk" and new_label == "high risk") or (
            old_label == "high risk" and new_label == "high risk" and new_score > old_score
        )

        if is_increase:
            modified_risk_deltas.append({
                "old_clause": item["old_clause"],
                "new_clause": item["new_clause"],
                "similarity": item["similarity"],
                "old_risk_label": old_label,
                "old_risk_score": old_score,
                "new_risk_label": new_label,
                "new_risk_score": new_score,
                "diff": clause_word_diff(item["old_clause"], item["new_clause"]),
            })

    return {
        "added": added,
        "removed": removed,
        "modified": modified,
        "unchanged": unchanged,
        "structural_changes": structural_changes,
        "added_high_risk": added_risks,
        "modified_risk_increase": modified_risk_deltas,
        "diagnostics": diagnostics,
    }


In [28]:
# print("nda1_clauses", original_clauses)
# print("nda2_clauses", altered_clauses)


def print_version_report(title, report):
    print(title)
    print("Added clauses:", len(report["added"]))
    print("Removed clauses:", len(report["removed"]))
    print("Modified clauses:", len(report["modified"]))
    print("Unchanged matched clauses:", len(report["unchanged"]))
    print("Structural merge/split changes:", len(report["structural_changes"]))
    print()
    print("Newly added high-risk clauses:", len(report["added_high_risk"]))
    print("Modified clauses with risk increase:", len(report["modified_risk_increase"]))
    print("\n=== MATCHING DIAGNOSTICS ===")
    print(report["diagnostics"])


version_report_baseline = analyze_version_changes(
    original_clauses,
    altered_clauses,
    hybrid=False,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
)

version_report_improved = analyze_version_changes(
    original_clauses,
    altered_clauses,
    hybrid=True,
    modified_threshold=0.50,
    candidate_floor=0.28,
    structural_threshold=0.36,
    word_char_weight=0.58,
)

print_version_report("=== BASELINE (word TF-IDF) ===", version_report_baseline)
print()
print_version_report("=== IMPROVED (hybrid word + char TF-IDF) ===", version_report_improved)

print("\n=== IMPROVED: ADDED HIGH-RISK CLAUSES (sample) ===")
for i, item in enumerate(version_report_improved["added_high_risk"][:3], 1):
    print(f"[{i}] score={item['risk_score']:.3f}")
    print(item["clause"][:300], "\n")

print("\n=== IMPROVED: STRUCTURAL CHANGES (sample) ===")
for i, item in enumerate(version_report_improved["structural_changes"][:3], 1):
    print(f"[{i}] new clause:", item["new_clause"][:220])
    for j, linked in enumerate(item["linked_old_clauses"], 1):
        print(f"   linked old {j} sim={linked['similarity']:.3f}:", linked["old_clause"][:180])
    print()

print("\n=== IMPROVED: MODIFIED CLAUSES WITH RISK INCREASE (sample) ===")
for i, item in enumerate(version_report_improved["modified_risk_increase"][:3], 1):
    print(f"[{i}] similarity={item['similarity']:.3f}")
    print(f"old: {item['old_risk_label']} ({item['old_risk_score']:.3f})")
    print(f"new: {item['new_risk_label']} ({item['new_risk_score']:.3f})")
    print("old clause:", item["old_clause"][:220])
    print("new clause:", item["new_clause"][:220])
    print("word diff:")
    print(item["diff"][:1200])
    print()


=== VERSION CHANGE SUMMARY ===
Added clauses: 3
Removed clauses: 233
Modified clauses: 8
Unchanged matched clauses: 8
Structural merge/split changes: 0

Newly added high-risk clauses: 3
Modified clauses with risk increase: 2

=== MATCHING DIAGNOSTICS ===
{'matched_pairs': 16, 'candidate_pairs': 31, 'threshold': 0.58, 'candidate_floor': 0.35, 'structural_threshold': 0.42}

=== ADDED HIGH-RISK CLAUSES (sample) ===
[1] score=0.800
d) with anyone, including other students, outside of the course. 

[2] score=0.708
11. In the event of breach, Discloser may seek injunctive relief in addition to any other remedies available at law or in equity. 

[3] score=0.684
12. UT will ensure that access to Confidential Information is restricted to Students and instructional personnel with a need to know, and that such access is revoked at course completion. This Agreement is made effective on the Effective Date. By:______________________________________ Title:________ 


=== STRUCTURAL CHANGES (sample) =